# Investigate parameter sensitivities in the model

In [1]:
import numpy as np
from matplotlib.cm import get_cmap
import matplotlib.pyplot as plt
import matplotlib as mpl

import multiprocessing as mp
from tqdm import tqdm
from joblib import Parallel, delayed
import os
from stoch_sim_model import *
from joint_plot_grid_nb import *
import seaborn as sns

plt.style.use('custom.mplstyle')
%config InlineBackend.figure_format = 'retina'

KeyboardInterrupt: 

## (1) Simulate model over sampled parameter space

In [ ]:
infection_type = "prim" #"prim", "sec"
sim_type = "agent" #"pop_ode", "agent"
reg_model = "mwc_like"

### Vary $d_I$

In [ ]:
# run simulations to see effect of alpha:
dIs = sample_grid(d = 1, l_bounds = 0, u_bounds = 1.0, runs = 20)

run_dIs = np.vstack(Parallel(n_jobs=20, batch_size = max(int(len(dIs)/20),1))(delayed(sum_sim)(d_I = param,
                                                                                                infection = infection_type,
                                                                                                vir_model = 'dep_harm',
                                                                                                sim_kind = sim_type,
                                                                                                reg_model = reg_model)
                                                    for param in dIs))

In [ ]:
plt.scatter(dIs, run_dIs[:,-16]/S_0)
plt.axvline(x = run_dIs[:,4][np.argmax(run_dIs[:,-16])], color = 'k', linestyle = '--')
plt.xlabel('harmfulness, '+param_names[4])
plt.ylabel('primary infection excess cell death \n'+stat_names[6])
print(run_dIs[:,4][np.argmax(run_dIs[:,-16])])

In [ ]:
plt.scatter(dIs, run_dIs[:,-12]/K_IE)
plt.axvline(x = run_dIs[:,4][np.argmax(run_dIs[:,-12])], color = 'k', linestyle = '--')
plt.xlabel('harmfulness, '+param_names[4])
plt.ylabel('primary infection response \n'+stat_names[-12])
#plt.semilogy()
print(run_dIs[:,4][np.argmax(run_dIs[:,-12])])

### Vary $\alpha$

In [ ]:
# run simulations to see effect of alpha:
alphas = sample_grid(d = 1, l_bounds = 0.1, u_bounds = 0.9,runs = 50)

run_alpha = np.vstack(Parallel(n_jobs=20, batch_size = max(int(len(alphas)/20),1))(delayed(sum_sim)(signal_weight = param,
                                                                                                     infection = infection_type,
                                                                                                     vir_model = 'dep_harm',
                                                                                                     sim_kind = sim_type,
                                                                                                     reg_model = reg_model)
                                                    for param in alphas))

In [ ]:
plt.scatter(alphas, run_alpha[:,-16]/S_0)
plt.xlabel(r'antigenic signal weight, $\alpha$')
plt.ylabel('primary infection excess cell death \n'+stat_names[6])

In [ ]:
plt.scatter(alphas, run_alpha[:,-12]/K_IE)
plt.xlabel(r'antigenic signal weight, $\alpha$')
plt.ylabel('primary infection response \n'+stat_names[-12])
#plt.semilogy()

### Sensitivities for other parameters

In [ ]:
# Define sampled values of parameters
runs = 100
rng = 2
infection_type = "prim" #"prim", "sec"
sim_type = "agent" #"pop_ode", "agent"
reg = np.array([1.0, 1.0, -1.0, -1.0, 1.0, 1.0]) # do this for a fixed network, to start

default_vals = np.array([S_0, I_0, b_I, d_S, d_I, d_IE, d_IH, K_IE, K_IH,
                  Aout_0, b_Ain, b_H, d_H, K_Ain, K_HE,
                  N_0, max_Na, b_myc, d_myc, myc_thresh,
                  t_act, t_unbind, t_Na_div, t_E_div, t_cM_div, t_eM_diff, t_E_out, t_E_die, t_E_cyt,
                  n_act, n_unbind, n_Na_div, n_E_div, n_cM_div, n_eM_diff, n_E_out, n_E_die, n_E_cyt])

sample_params = sample_grid(d = 10,
                            l_bounds = default_vals[[2,3,4,6,7,8,11,13,14,17]]/rng, 
                            u_bounds = default_vals[[2,3,4,6,7,8,11,13,14,17]]*[rng, rng, rng, rng, rng, rng, rng, rng, 5/3, rng],
                            runs = runs)

In [ ]:
# Collect simulation results across samples
#os.cpu_count()
sample_data = np.vstack(Parallel(n_jobs= 35, batch_size = max(int(runs/35),1))(delayed(sum_sim)(b_I = p[0], 
                                                                                              d_S = p[1], 
                                                                                              d_IE = p[2], 
                                                                                              d_IH = p[3], 
                                                                                              K_IE = p[4], 
                                                                                              K_IH = p[5],
                                                                                              b_H = p[6], 
                                                                                              K_Ain = p[7], 
                                                                                              K_HE = p[8],
                                                                                              b_myc = p[9], 
                                                                                              regulation_coeffs = reg,
                                                                                              infection = infection_type)
                                                    for p in sample_params))

### (2) Compute parameter dependencies using linear regression

In [ ]:
# import packages: will use sparse regresssion
from sklearn.metrics import mutual_info_score
from sklearn.linear_model import LinearRegression
from sklearn import linear_model

parameters = sample_data[:, 0:len(param_names)]
out_data = sample_data[:, len(param_names):]

In [ ]:
# run sparse regressions and store coefficient data
coeff_data = np.zeros((out_data.shape[1]-6, parameters.shape[1]))

for k, stat in enumerate(stat_names[6:]):
    #lr = linear_model.Lasso(alpha=0.1)
    lr = LinearRegression()
    lr.fit(np.log(parameters/default_vals), out_data[:,k + 6].reshape(-1,1)) # X = log-scaled-parameter values, centered at default value
    coeff_data[k,:] = lr.coef_/lr.intercept_ # scale coefficients by value at default value to give interpretability.

In [ ]:
# plot results: what matters is magnitude, although the sign gives color and should hopefully match our intuition about the model
fig, axs = plt.subplots(int((out_data.shape[1]-6)), 1, figsize=(25, 80))
r = np.arange(parameters.shape[1])
width = 0.5

for i, ax in enumerate(axs.flat):
    ax.bar(r, coeff_data[i,:], width = width)
    ax.axhline(y = 0.01, color = 'r', linestyle = '--')
    #ax.set_yscale('log')
    ax.set_xticks(r)
    ax.set_xticklabels(param_names, fontsize = 12)
    ax.set(ylabel=stat_names[6:][i])
    #ax.label_outer()